# Простой baseline: линейная регрессия

Этот блокнот можно выдать ученикам как стартовый код, если цель занятия — проверять не загрузку данных и базовую предобработку, а улучшение модели.

Baseline делает минимум:

- читает train/test;
- кодирует категории one-hot;
- заполняет пропуски;
- обучает `LinearRegression`;
- сохраняет `submission.csv`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_DIR = Path("data")


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

target = "price_mln"
id_col = "id"

features = [c for c in train.columns if c not in [target, id_col]]
X = train[features]
y = train[target]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)


In [ ]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", make_ohe()),
                ]
            ),
            categorical_features,
        ),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LinearRegression()),
    ]
)

model.fit(X_train, y_train)
val_pred = model.predict(X_val)
print("Validation MAE:", round(mean_absolute_error(y_val, val_pred), 3))


In [ ]:
model.fit(X, y)
test_pred = model.predict(test[features])

submission = pd.DataFrame(
    {
        "id": test[id_col],
        "price_mln": np.maximum(test_pred, 0).round(3),
    }
)

submission.to_csv("submission.csv", index=False)
submission.head()
